In [1]:
import pandas as pd

df = pd.read_parquet("../data/raw/mta_hourly/2023-01.parquet")
stations = pd.read_csv("../data/raw/stations.csv", dtype={"station_complex_id": str})

print(df.shape)
print(df.dtypes)
display(df.head())

# суммарный пассажиропоток по дням: будни около 3,5–4 млн, выходные меньше
daily = df.groupby(df["transit_timestamp"].dt.date)["ridership"].sum()
display(daily)

# самые загруженные станции за месяц
top = (df.groupby("station_complex_id")["ridership"].sum()
         .nlargest(10).reset_index()
         .merge(stations[["station_complex_id", "station_complex"]], on="station_complex_id"))
display(top)

(323082, 4)
transit_timestamp     datetime64[us]
station_complex_id               str
ridership                      int32
transfers                      int32
dtype: object


,transit_timestamp,station_complex_id,ridership,transfers
0,2023-01-01,1,79,0
1,2023-01-01,10,131,0
2,2023-01-01,100,13,0
3,2023-01-01,101,147,0
4,2023-01-01,103,153,0


transit_timestamp
2023-01-01    1672056
2023-01-02    1950083
2023-01-03    3196136
2023-01-04    3433910
2023-01-05    3448521
2023-01-06    3248384
2023-01-07    2115891
2023-01-08    1610895
2023-01-09    3164830
2023-01-10    3489623
2023-01-11    3529752
2023-01-12    3492411
2023-01-13    3288923
2023-01-14    2072434
2023-01-15    1704364
2023-01-16    2184679
2023-01-17    3471080
2023-01-18    3624821
2023-01-19    3513842
2023-01-20    3395007
2023-01-21    2136946
2023-01-22    1594148
2023-01-23    3155381
2023-01-24    3588188
2023-01-25    3532395
2023-01-26    3653799
2023-01-27    3357916
2023-01-28    2250053
2023-01-29    1625433
2023-01-30    3244394
2023-01-31    3679833
Name: ridership, dtype: int32

,station_complex_id,ridership,station_complex
0,611,3127242,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)"
1,610,2202586,"Grand Central-42 St (S,4,5,6,7)"
2,607,1812364,"34 St-Herald Sq (B,D,F,M,N,Q,R,W)"
3,602,1596271,"14 St-Union Sq (L,N,Q,R,W,4,5,6)"
4,628,1308342,"Fulton St (A,C,J,Z,2,3,4,5)"
5,164,1297376,"34 St-Penn Station (A,C,E)"
6,318,1210512,"34 St-Penn Station (1,2,3)"
7,614,1162355,"59 St-Columbus Circle (A,B,C,D,1)"
8,447,1064270,Flushing-Main St (7)
9,616,1048168,"74-Broadway (7)/Jackson Hts-Roosevelt Av (E,F,..."


In [2]:
df.duplicated(["transit_timestamp", "station_complex_id"]).sum()

np.int64(12015)

In [3]:
import pandas as pd
from pathlib import Path

files = sorted(Path("../data/raw/mta_hourly").glob("20*.parquet"))
df = pd.concat(pd.read_parquet(f) for f in files)
stations = pd.read_csv("../data/raw/stations.csv", dtype={"station_complex_id": str})

# в какие дни у станции были данные
days = (df.assign(day=df["transit_timestamp"].dt.normalize())
          .groupby(["station_complex_id", "day"])["ridership"].sum().reset_index())
all_days = pd.date_range("2023-01-01", "2026-09-24", freq="D")

rows = []
for sid, g in days.groupby("station_complex_id"):
    missing = all_days.difference(g["day"])
    if len(missing) == 0:
        continue
    # склеиваем подряд идущие отсутствующие дни в отрезки
    grp = (missing.to_series().diff() != pd.Timedelta(days=1)).cumsum()
    for _, seg in missing.to_series().groupby(grp):
        rows.append({"station_complex_id": sid, "from": seg.min().date(),
                     "to": seg.max().date(), "days": len(seg)})

gaps = (pd.DataFrame(rows)
          .merge(stations[["station_complex_id", "station_complex", "borough"]],
                 on="station_complex_id")
          .sort_values(["days"], ascending=False))
pd.set_option("display.max_rows", 200, "display.max_colwidth", 60)
display(gaps)
print(f"Станций с пропусками: {gaps['station_complex_id'].nunique()}")

,station_complex_id,from,to,days,station_complex,borough
563,502,2023-02-25,2026-09-24,1308,Tompkinsville (SIR),Staten Island
559,501,2023-01-01,2026-06-17,1264,St George (SIR),Staten Island
138,202,2025-01-18,2025-05-18,121,"Beach 105 St (A,S)",Queens
146,203,2025-01-21,2025-05-16,116,"Rockaway Park-Beach 116 St (A,S)",Queens
222,208,2025-01-26,2025-05-14,109,Beach 25 St (A),Queens
...,...,...,...,...,...,...
716,91,2026-08-16,2026-08-16,1,"Alabama Av (J,Z)",Brooklyn
715,91,2026-08-09,2026-08-09,1,"Alabama Av (J,Z)",Brooklyn
714,91,2025-03-23,2025-03-23,1,"Alabama Av (J,Z)",Brooklyn
6,107,2023-01-29,2023-01-29,1,"Broad St (J,Z)",Manhattan


Станций с пропусками: 126


In [4]:
gaps = gaps[~gaps["station_complex_id"].isin(["501", "502"])]

bins = [0, 1, 2, 7, 30, 10_000]
labels = ["1 день", "2 дня", "3–7 дней", "8–30 дней", ">30 дней"]
gaps["длина"] = pd.cut(gaps["days"], bins=bins, labels=labels)
display(gaps.groupby("длина", observed=False)
            .agg(отрезков=("days", "size"), станций=("station_complex_id", "nunique")))

# всё, что длиннее 2 дней, — по отдельности
display(gaps[gaps["days"] > 2].sort_values(["from", "station_complex"]))


,отрезков,станций
длина,,
1 день,511,119
2 дня,135,59
3–7 дней,50,18
8–30 дней,15,10
>30 дней,13,9


,station_complex_id,from,to,days,station_complex,borough,длина
279,247,2023-05-27,2023-05-29,3,Bay Pkwy (F),Brooklyn,3–7 дней
22,111,2023-08-31,2023-09-02,3,Seneca Av (M),Queens,3–7 дней
43,114,2023-09-01,2023-09-04,4,Central Av (M),Brooklyn,3–7 дней
383,253,2024-02-23,2024-02-25,3,Neptune Av (F),Brooklyn,3–7 дней
534,446,2024-04-22,2024-04-24,3,Morris Park (5),Bronx,3–7 дней
409,282,2024-06-29,2024-07-01,3,21 St (G),Queens,3–7 дней
410,282,2024-07-04,2024-07-11,8,21 St (G),Queens,8–30 дней
429,284,2024-07-08,2024-07-10,3,Nassau Av (G),Brooklyn,3–7 дней
437,286,2024-07-09,2024-07-11,3,Broadway (G),Brooklyn,3–7 дней
430,284,2024-07-12,2024-07-14,3,Nassau Av (G),Brooklyn,3–7 дней
